In [0]:
USE CATALOG tibia_analytics;
USE SCHEMA bronze;
SET TIME ZONE 'UTC';

In [0]:
COPY INTO bronze.raw_highscores
FROM (
  SELECT
    sha1(concat_ws('|', _metadata.file_path, cast(_metadata.file_modification_time AS STRING))) AS raw_highscores_id,
    from_json(
      to_json(highscores),
      'STRUCT<
        world: STRING,
        category: STRING,
        vocation: STRING,
        highscore_age: INT,
        highscore_list: ARRAY<STRUCT<
          rank: INT,
          name: STRING,
          title: STRING,
          vocation: STRING,
          world: STRING,
          level: INT,
          value: INT
        >>,
        highscore_page: STRUCT<
          current_page: INT,
          total_pages: INT,
          total_records: INT
        >
      >'
    ) AS highscores,
    from_json(
      to_json(information),
      'STRUCT<
        api: STRUCT<
          version: INT,
          release: STRING,
          commit: STRING
        >,
        timestamp: STRING,
        tibia_urls: ARRAY<STRING>,
        status: STRUCT<
          error: INT,
          http_code: INT,
          message: STRING
        >
      >'
    ) AS information,
    cast(regexp_extract(_metadata.file_path, '([0-9]{4}-[0-9]{2}-[0-9]{2})', 1) AS DATE) AS source_file_date,
    _metadata.file_path AS source_file_path,
    _metadata.file_modification_time AS source_file_modified_at,
    current_timestamp() AS ingested_at
  FROM '/Volumes/tibia_analytics/bronze/raw/highscores/*'
)
FILEFORMAT = JSON
FORMAT_OPTIONS ('multiLine' = 'true')
COPY_OPTIONS ('mergeSchema' = 'false');